In [ ]:
///- --package .

#### network (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:= one () : i32 = 256
:= input_count () : i32 = 8
:= hidden_count () : i32 = 6
:= output_count () : i32 = 3
:= attacker_unit () : i32 = 85
:= weight_count () : i32 = input_count () * hidden_count () + hidden_count () * output_count ()

:! weights = array i32

:== sensed = { health : i32; attackers : i32; target : i32; distance : i32; heal : i32; strike : i32; flees : i32 }

:! inputs = sensed

:| feature =
    | HealthFraction
    | Attackers
    | TargetHealth
    | TargetDistance
    | HealReady
    | StrikeReady
    | TargetFleeing
    | Bias

:| output =
    | HealOutput
    | EngageOutput
    | RetreatOutput

:! hidden = i32

:= feature_slot (f : feature) : i32 =
    ?= f ?|
    | HealthFraction => 0
    | Attackers => 1
    | TargetHealth => 2
    | TargetDistance => 3
    | HealReady => 4
    | StrikeReady => 5
    | TargetFleeing => 6
    | Bias => 7

:= output_slot (o : output) : i32 =
    ?= o ?|
    | HealOutput => 0
    | EngageOutput => 1
    | RetreatOutput => 2

:= input_weight_slot (hidden h) (f : feature) : i32 = h * input_count () + feature_slot f

:= output_weight_slot (o : output) (hidden h) : i32 =
    input_count () * hidden_count () + output_slot o * hidden_count () + h

:== decision = { heal : bool; engage : bool; retreat : bool }

:= filled (count : i32) (value : i32) : array i32 =
    := x : array i32 = create count
    :~ ^ fill (i : i32) : () =
        ? i < count ?>
            set x i value
            fill (i + 1)
    fill 0
    x

:= zero_weights () : weights = weights (filled (weight_count ()) 0)

:= weight (weights w) (i : i32) : i32 = index w i

:= feature_flag (b : bool) : i32 = ? b ?> one () |: 0

:= sensed_distance (distance : i32) : i32 = (? distance > 7 ?> 7 |: distance) * 32

:= flags_of (heal : bool) (strike : bool) (flees : bool) : i32 * i32 * i32 =
    feature_flag heal, feature_flag strike, feature_flag flees

:= sensed_of (h : i32) (a : i32) (t : i32) (d : i32) ((x, y, z) : i32 * i32 * i32) : sensed =
    {
        health = h; attackers = a * attacker_unit (); target = t; distance = sensed_distance d
        heal = x; strike = y; flees = z
    }

:= inputs_of
    (health : i32) (attackers : i32) (target : i32) (distance : i32) (heal : bool) (strike : bool) (flees : bool)
    : inputs
    =
    inputs (sensed_of health attackers target distance (flags_of heal strike flees))

:= sensed_dot (coef : i32 -> i32) ((inputs s) : inputs) : i32 =
    coef 0 * s::health + coef 1 * s::attackers + coef 2 * s::target + coef 3 * s::distance

:= flag_dot (coef : i32 -> i32) ((inputs s) : inputs) : i32 =
    coef 4 * s::heal + coef 5 * s::strike + coef 6 * s::flees + coef 7 * one ()

:= dot_with (coef : i32 -> i32) (x : inputs) : i32 = sensed_dot coef x + flag_dot coef x

:= term (w : weights) (j : i32) (i : i32) (value : i32) : i32 = weight w (j * input_count () + i) * value

:= sensed_terms (w : weights) ((inputs s) : inputs) (j : i32) : i32 =
    term w j 0 s::health + term w j 1 s::attackers + term w j 2 s::target + term w j 3 s::distance

:= flag_terms (w : weights) ((inputs s) : inputs) (j : i32) : i32 =
    term w j 4 s::heal + term w j 5 s::strike + term w j 6 s::flees + term w j 7 (one ())

:= hidden_unit (w : weights) (x : inputs) (j : i32) : i32 =
    := h = (sensed_terms w x j + flag_terms w x j) / one ()
    ? h > 0 ?> h |: 0

:= hidden_layer (w : weights) (x : inputs) : array i32 =
    := hidden = filled (hidden_count ()) 0
    :~ ^ activate (j : i32) : () =
        ? j < hidden_count () ?>
            set hidden j (hidden_unit w x j)
            activate (j + 1)
    activate 0
    hidden

:= output_weight_at (w : weights) (k : i32) (j : i32) : i32 =
    weight w (input_count () * hidden_count () + k * hidden_count () + j)

:~ ^ output_sum (w : weights) (hidden : array i32) (k : i32) (j : i32) (total : i32) : i32 =
    ? j >= hidden_count () ?> total |: output_sum w hidden k (j + 1) (total + output_weight_at w k j * index hidden j)

:= output_unit (w : weights) (hidden : array i32) (k : i32) : i32 = output_sum w hidden k 0 0 / one ()

:= decision_of (heal : i32) (engage : i32) (retreat : i32) : decision =
    { heal = heal > 0; engage = engage > 0 && engage >= retreat; retreat = retreat > 0 && retreat > engage }

:= scaled (heal : i32) (engage : i32) (retreat : i32) : decision =
    decision_of (heal / one ()) (engage / one ()) (retreat / one ())

:= added (w : weights) (j : i32) (a : i32) (k : i32) (acc : i32) : i32 = acc + output_weight_at w k j * a

:~ ^ outputs_from (w : weights) (x : inputs) (j : i32) (h : i32) (e : i32) (r : i32) : decision =
    ? j >= hidden_count ()
    ?> scaled h e r
    |:
        := a = hidden_unit w x j
        outputs_from w x (j + 1) (added w j a 0 h) (added w j a 1 e) (added w j a 2 r)

:= decide (w : weights) (x : inputs) : decision = outputs_from w x 0 0 0 0

:= with_weight (i : i32) (value : i32) (weights w) : weights =
    set w i value
    weights w

:= with_input_weight (h : hidden) (f : feature) (value : i32) (w : weights) : weights =
    with_weight (input_weight_slot h f) value w

:= with_output_weight (o : output) (h : hidden) (value : i32) (w : weights) : weights =
    with_weight (output_weight_slot o h) value w

:= fraction_below (heal_below : i32) (max_health : i32) : i32 = (heal_below - 1) * one () / max_health + 1

:= with_heal_threshold (heal_below_fraction : i32) (w : weights) : weights =
    w
    |> with_input_weight (hidden 0) HealthFraction (0 - one ())
    |> with_input_weight (hidden 0) Bias heal_below_fraction
    |> with_output_weight HealOutput (hidden 0) (one ())

:= with_engaging (w : weights) : weights =
    w |> with_input_weight (hidden 1) Bias (one ()) |> with_output_weight EngageOutput (hidden 1) (one ())

:= threshold_healer (heal_below_fraction : i32) : weights =
    zero_weights () |> with_heal_threshold heal_below_fraction |> with_engaging

:= with_kite_trigger (w : weights) : weights =
    w |> with_input_weight (hidden 2) Attackers (one ()) |> with_input_weight (hidden 2) StrikeReady (0 - one ())

:= with_kiting (min_attackers : i32) (w : weights) : weights =
    w
    |> with_kite_trigger
    |> with_input_weight (hidden 2) Bias (0 - (min_attackers - 1) * attacker_unit ())
    |> with_output_weight RetreatOutput (hidden 2) (4 * one ())

:= clamp_weight (x : i32) : i32 = ? x < -2048 ?> -2048 |? x > 2048 ?> 2048 |: x

:== weight_search = { seed : u32; generations : i32; offspring : i32; spread : i32 }

:== scored = { w : weights; score : i32 }

:== weight_round = { s : weight_search; index : i32; width : i32; parent : scored }

:< keyed_rng

:| network_roll =
    | WeightNudge : i32

:+ purpose_code network_roll =
    \ r =>
        ?= r ?|
        | WeightNudge i => conv (1000 + i)

:= weight_key (r : weight_round) (child : i32) (i : i32) : keyed_rng::roll_key network_roll =
    := s = keyed_rng::seed r::s::seed
    keyed_rng::roll_key s (keyed_rng::tick (conv r::index)) (keyed_rng::entity (conv child)) (WeightNudge i)

:= weight_draw (r : weight_round) (child : i32) (i : i32) : i32 =
    conv (keyed_rng::roll_between 0 (conv (2 * r::width)) (weight_key r child i))

:~ ^ copy_mutated (r : weight_round) (child : i32) (next : array i32) (i : i32) : () =
    ? i < weight_count () ?>
        set next i (clamp_weight (weight r::parent::w i + weight_draw r child i - r::width))
        copy_mutated r child next (i + 1)

:= mutate (r : weight_round) (child : i32) : weights =
    := next = filled (weight_count ()) 0
    copy_mutated r child next 0
    weights next

:= fitter_weights (fitness : weights -> i32) (candidate : weights) (best : scored) : scored =
    := score = fitness candidate
    ? score > best::score ?> { w = candidate; score = score } |: best

:~ ^ offspring_from (fitness : weights -> i32) (r : weight_round) (child : i32) (best : scored) : scored =
    ? child >= r::s::offspring
    ?> best
    |: offspring_from fitness r (child + 1) (fitter_weights fitness (mutate r child) best)

:= next_weight_round (r : weight_round) (best : scored) : weight_round =
    { r ?| index = r::index + 1; width = (? r::width > 4 ?> r::width * 4 / 5 |: 4); parent = best }

:~ ^ weight_climb (fitness : weights -> i32) (r : weight_round) : weights =
    ? r::index >= r::s::generations
    ?> r::parent::w
    |: weight_climb fitness (next_weight_round r (offspring_from fitness r 0 r::parent))

:= evolve (fitness : weights -> i32) (s : weight_search) (start : weights) : weights =
    weight_climb fitness { s = s; index = 0; width = s::spread; parent = { w = start; score = fitness start } }

:? judge f : f -> weights -> i32

:= fitter_by \* f {judge}. (j : f) (candidate : weights) (best : scored) : scored =
    := score = judge j candidate
    ? score > best::score ?> { w = candidate; score = score } |: best

:~ ^ offspring_by \* f {judge}. (j : f) (r : weight_round) (child : i32) (best : scored) : scored =
    ? child >= r::s::offspring ?> best |: offspring_by j r (child + 1) (fitter_by j (mutate r child) best)

:~ ^ climb_by \* f {judge}. (j : f) (r : weight_round) : weights =
    ? r::index >= r::s::generations ?> r::parent::w |: climb_by j (next_weight_round r (offspring_by j r 0 r::parent))

:= evolve_by \* f {judge}. (j : f) (s : weight_search) (start : weights) : weights =
    climb_by j { s = s; index = 0; width = s::spread; parent = { w = start; score = judge j start } }

:= scores_by \* f {judge}. (j : f) (r : weight_round) : array i32 =
    parallel::map_index r::s::offspring \ child => judge j (mutate r child)

:= fitter_scored (r : weight_round) (scores : array i32) (child : i32) (best : scored) : scored =
    := score = index scores child
    ? score > best::score ?> { w = mutate r child; score = score } |: best

:~ ^ best_scored (r : weight_round) (scores : array i32) (child : i32) (best : scored) : scored =
    ? child >= r::s::offspring ?> best |: best_scored r scores (child + 1) (fitter_scored r scores child best)

:~ ^ climb_par_by \* f {judge}. (j : f) (r : weight_round) : weights =
    ? r::index >= r::s::generations
    ?> r::parent::w
    |: climb_par_by j (next_weight_round r (best_scored r (scores_by j r) 0 r::parent))

:= evolve_par_by \* f {judge}. (j : f) (s : weight_search) (start : weights) : weights =
    climb_par_by j { s = s; index = 0; width = s::spread; parent = { w = start; score = judge j start } }

:| elites =
    | TopK : i32
    | AboveMean

:~ ^ score_total (scores : array i32) (n : i32) (i : i32) (total : i64) : i64 =
    ? i >= n ?> total |: score_total scores n (i + 1) (total + conv (index scores i))

:= ranks_above (scores : array i32) (child : i32) (i : i32) : bool =
    index scores i > index scores child || (index scores i = index scores child && i < child)

:~ ^ rank_of (scores : array i32) (child : i32) (i : i32) (rank : i32) : i32 =
    ? i >= length scores
    ?> rank
    |: rank_of scores child (i + 1) (? ranks_above scores child i ?> rank + 1 |: rank)

:= is_elite (e : elites) (scores : array i32) (child : i32) : bool =
    ?= e ?|
    | AboveMean => conv (index scores child) * conv (length scores) >= score_total scores (length scores) 0 0i64
    | TopK k => rank_of scores child 0 0 < k

:~ ^ add_weights (sums : array i32) (w : weights) (i : i32) : () =
    ? i < weight_count () ?>
        set sums i (index sums i + weight w i)
        add_weights sums w (i + 1)

:= take_elite (sums : array i32) (w : weights) (count : i32) : i32 =
    add_weights sums w 0
    count + 1

:== elite_pass = { e : elites; r : weight_round; scores : array i32; sums : array i32 }

:= take_if_elite (p : elite_pass) (child : i32) (count : i32) : i32 =
    ? is_elite p::e p::scores child ?> take_elite p::sums (mutate p::r child) count |: count

:~ ^ sum_elites (p : elite_pass) (child : i32) (count : i32) : i32 =
    ? child >= p::r::s::offspring ?> count |: sum_elites p (child + 1) (take_if_elite p child count)

:~ ^ copy_mean (sums : array i32) (count : i32) (next : array i32) (i : i32) : () =
    ? i < weight_count () ?>
        set next i (clamp_weight (index sums i / count))
        copy_mean sums count next (i + 1)

:= mean_weights (sums : array i32) (count : i32) : weights =
    := next = filled (weight_count ()) 0
    copy_mean sums count next 0
    weights next

:| spread =
    | Shrink
    | EliteSpread

:== cem_rule = { pick : elites; spread : spread }

:= elite_pass_of (e : elites) (r : weight_round) (scores : array i32) : elite_pass =
    { e = e; r = r; scores = scores; sums = filled (weight_count ()) 0 }

:= elite_mean (p : elite_pass) (count : i32) : weights = ? count = 0 ?> p::r::parent::w |: mean_weights p::sums count

:= distance (a : i32) (b : i32) : i32 = ? a > b ?> a - b |: b - a

:~ ^ abs_dev (w : weights) (mean : weights) (i : i32) (total : i32) : i32 =
    ? i >= weight_count () ?> total |: abs_dev w mean (i + 1) (total + distance (weight w i) (weight mean i))

:= dev_if_elite (p : elite_pass) (mean : weights) (child : i32) (total : i32) : i32 =
    ? is_elite p::e p::scores child ?> abs_dev (mutate p::r child) mean 0 total |: total

:~ ^ sum_devs (p : elite_pass) (mean : weights) (child : i32) (total : i32) : i32 =
    ? child >= p::r::s::offspring ?> total |: sum_devs p mean (child + 1) (dev_if_elite p mean child total)

:= elite_width (p : elite_pass) (mean : weights) (count : i32) : i32 =
    := spread = sum_devs p mean 0 0 / max 1 (count * weight_count ())
    max 4 ((p::r::width + spread) / 2)

:= next_width (c : cem_rule) (p : elite_pass) (mean : weights) (count : i32) : i32 =
    ?= c::spread ?|
    | Shrink => (next_weight_round p::r p::r::parent)::width
    | EliteSpread => elite_width p mean count

:== cem_step = { mean : scored; width : i32 }

:= cem_step_of \* f {judge}. (j : f) (c : cem_rule) (p : elite_pass) (count : i32) : cem_step =
    := mean = elite_mean p count
    { mean = { w = mean; score = judge j mean }; width = next_width c p mean count }

:= cem_step_by \* f {judge}. (j : f) (c : cem_rule) (r : weight_round) : cem_step =
    := p = elite_pass_of c::pick r (scores_by j r)
    cem_step_of j c p (sum_elites p 0 0)

:= cem_round (r : weight_round) (step : cem_step) : weight_round =
    { r ?| index = r::index + 1; width = step::width; parent = step::mean }

:= kept_best (next : scored) (best : scored) : scored = ? next::score > best::score ?> next |: best

:~ ^ climb_cem_by \* f {judge}. (j : f) (c : cem_rule) (r : weight_round) (best : scored) : weights =
    ? r::index >= r::s::generations
    ?> best::w
    |:
        := step = cem_step_by j c r
        climb_cem_by j c (cem_round r step) (kept_best step::mean best)

:= evolve_cem_by \* f {judge}. (j : f) (c : cem_rule) (s : weight_search) (start : weights) : weights =
    := first = { w = start; score = judge j start }
    climb_cem_by j c { s = s; index = 0; width = s::spread; parent = first } first

:= vetted (v : weights -> i32) (s : scored) : scored = { w = s::w; score = v s::w }

:= selected_step \* f {judge}. (j : f) (v : weights -> i32) (r : weight_round) (b : scored) : weight_round * scored =
    := next = best_scored r (scores_by j r) 0 r::parent
    next_weight_round r next, kept_best (vetted v next) b

:~ ^ climb_selected_by \* f {judge}. (j : f) (v : weights -> i32) (r : weight_round) (b : scored) : weights =
    ? r::index >= r::s::generations
    ?> b::w
    |:
        := r2, b2 = selected_step j v r b
        climb_selected_by j v r2 b2

:= first_round \* f {judge}. (j : f) (s : weight_search) (start : weights) : weight_round =
    { s = s; index = 0; width = s::spread; parent = { w = start; score = judge j start } }

:= evolve_selected_by \* f {judge}. (j : f) (v : weights -> i32) (s : weight_search) (start : weights) : weights =
    climb_selected_by j v (first_round j s start) (vetted v { w = start; score = 0 })

#### a_zero_network_holds (test)

In [1]:
///- --test --symbol
///> c

:= ~health = 128i32
:= d = decide (zero_weights ()) (inputs_of health 1 200 1 !+ !+ !-)
d::heal |> _assert_eq !-
d::engage |> _assert_eq !-
d::retreat |> _assert_eq !-

__assert_eq { expected = false }
__assert_eq { expected = false }
__assert_eq { expected = false }

In [ ]:
:= hand_wired_healer () : weights =
    zero_weights ()
    |> with_input_weight (hidden 0) HealthFraction (-256)
    |> with_input_weight (hidden 0) Bias 128
    |> with_output_weight HealOutput (hidden 0) 256

:= heals_at (w : weights) (health : i32) : bool = (decide w (inputs_of health 0 0 0 !+ !+ !-))::heal

#### a_hand_wired_network_heals_when_low (test)

In [2]:
///- --test --symbol
///> c

:= ~health = 64i32
:= w = hand_wired_healer ()
(heals_at w health, heals_at w (health + 100)) |> _assert_eq (!+, !-)

__assert_eq { expected = (true, false) }

In [ ]:
:= threshold_at (health : i32) (attackers : i32) : decision =
    decide (threshold_healer 100) (inputs_of health attackers 0 1 !+ !+ !-)

#### a_threshold_healer_heals_below_and_engages_above (test)

In [3]:
///- --test --symbol
///> c

:= ~health = 99i32
:= above = threshold_at (health + 1) 0
((threshold_at health 0)::heal, above::heal, above::engage, (threshold_at (health + 1) 2)::retreat)
|> _assert_eq (!+, !-, !+, !-)

__assert_eq { expected = (true, false, true, false) }

In [ ]:
:= kiter_at (health : i32) (attackers : i32) (strike_ready : bool) : decision =
    decide (threshold_healer 100 |> with_kiting 2) (inputs_of health attackers 0 1 !+ strike_ready !-)

#### kiting_retreats_only_while_the_strike_recovers (test)

In [4]:
///- --test --symbol
///> c

:= ~health = 400i32
:= ready = kiter_at health 2 !+
((kiter_at health 2 !-)::retreat, ready::retreat, (kiter_at health 1 !-)::retreat, ready::engage)
|> _assert_eq (!+, !-, !-, !+)

__assert_eq { expected = (true, false, false, true) }

#### an_output_is_not_a_feature (test)

In [5]:
///- --test --symbol --fails "Unification failure"
///> c

zero_weights () |> with_input_weight (hidden 0) HealOutput 1

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/tibia/tibia/src/network.spi:433:53 (main.spi:399:53): ETY-TermError: Unification failure. Got:      (feature : *) Expected: (output : *) In the definition: spiral_cells_test.


#### inputs_are_not_weights (test)

In [6]:
///- --test --symbol --fails "Unification failure"
///> c

decide (inputs_of 1 0 0 0 !+ !+ !-) (inputs_of 1 0 0 0 !+ !+ !-)

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/tibia/tibia/src/network.spi:437:13 (main.spi:399:13): ETY-TermError: Unification failure. Got:      (inputs : *) Expected: (weights : *) In the definition: spiral_cells_test.


In [ ]:
:~ ^ heal_score (w : weights) (h : i32) (total : i32) : i32 =
    ? h > 256 ?> total |: heal_score w (h + 16) (? (h < 64) = heals_at w h ?> total + 1 |: total)

:= quarter_healer_search (seed : u32) : weight_search = { seed = seed; generations = 30; offspring = 12; spread = 256 }

#### evolution_learns_to_heal_below_a_quarter (test)

In [7]:
///- --test --symbol
///> c

:= ~seed = 5u32
:= score w = heal_score w 0 0
score (evolve score (quarter_healer_search seed) (zero_weights ())) |> _assert_eq 17i32

__assert_eq { expected = 17 }

#### network_decides_on_gpu (test)

In [8]:
///- --test --symbol
///> wgsl

:= ~health = 64i32
:= w = hand_wired_healer ()
? heals_at w health && heals_at w (health + 100) = !- ?> 0i32 |: 1i32

In [ ]:
:! heal_judge = i32

:+ judge heal_judge = \ (heal_judge _) w => heal_score w 0 0

:~ ^ weight_checksum (w : weights) (i : i32) (total : i32) : i32 =
    ? i >= weight_count () ?> total |: weight_checksum w (i + 1) (total + weight w i * (i + 1))

:= checksum_of (w : weights) : i32 = weight_checksum w 0 0

#### parallel_climb_keeps_the_sequential_winner (test)

In [9]:
///- --test --symbol
///> c
///> rust

:= ~seed = 5u32
:= s = quarter_healer_search seed
checksum_of (evolve_par_by (heal_judge 0) s (zero_weights ()))
|> _assert_eq (checksum_of (evolve_by (heal_judge 0) s (zero_weights ())))

.c (c)
__assert_eq { expected = -29232 }
.rs (rust)
__assert_eq { expected = -29232 }

0

In [ ]:
:= four_scores (a : i32) (b : i32) (c : i32) (d : i32) : array i32 =
    := x : array i32 = create 4
    set x 0 a
    set x 1 b
    set x 2 c
    set x 3 d
    x

:= elite_mask (e : elites) (scores : array i32) : bool * bool * bool * bool =
    is_elite e scores 0, is_elite e scores 1, is_elite e scores 2, is_elite e scores 3

:= three_masks (s : array i32) = elite_mask (TopK 1) s, elite_mask (TopK 2) s, elite_mask AboveMean s

#### tied_elites_go_to_the_earlier_child (test)

In [10]:
///- --test --symbol
///> c
///> rust

:= ~top = 5i32
three_masks (four_scores top top 3 1)
|> _assert_eq ((!+, !-, !-, !-), (!+, !+, !-, !-), (!+, !+, !-, !-))

.c (c)
__assert_eq { expected = ((true, false, false, false), (true, true, false, false), true, true, false, false) }
.rs (rust)
__assert_eq { expected = ((true, false, false, false), (true, true, false, false), true, true, false, false) }

0

#### the_mean_rule_does_not_overflow_on_large_scores (test)

In [11]:
///- --test --symbol
///> c
///> rust

:= ~low = -1000000000i32
elite_mask AboveMean (four_scores low low (low + 1) low) |> _assert_eq (!-, !-, !+, !-)

.c (c)
__assert_eq { expected = (false, false, true, false) }
.rs (rust)
__assert_eq { expected = (false, false, true, false) }

0

In [ ]:
:= cem_heal (c : cem_rule) (seed : u32) : i32 =
    heal_score (evolve_cem_by (heal_judge 0) c (quarter_healer_search seed) (zero_weights ())) 0 0

:= cem_never_below (spread : spread) (seed : u32) : bool =
    cem_heal { pick = AboveMean; spread = spread } seed >= heal_score (zero_weights ()) 0 0

#### cem_never_ends_below_its_start (test)

In [12]:
///- --test --symbol
///> c

:= ~seed = 5u32
(cem_never_below Shrink seed, cem_never_below EliteSpread seed) |> _assert_eq (!+, !+)

__assert_eq { expected = (true, true) }

In [ ]:
:= selected_heal (v : weights -> i32) (seed : u32) : weights =
    evolve_selected_by (heal_judge 0) v (quarter_healer_search seed) (zero_weights ())

:= climbed_heal (seed : u32) : i32 =
    heal_score (evolve_by (heal_judge 0) (quarter_healer_search seed) (zero_weights ())) 0 0

#### training_selection_keeps_the_climbs_score (test)

In [13]:
///- --test --symbol
///> c
///> rust

:= ~seed = 5u32
heal_score (selected_heal (\ w => heal_score w 0 0) seed) 0 0 |> _assert_eq (climbed_heal seed)

.c (c)
__assert_eq { expected = 17 }
.rs (rust)
__assert_eq { expected = 17 }

0

#### a_flat_validator_keeps_the_start (test)

In [14]:
///- --test --symbol
///> c
///> rust

:= ~seed = 5u32
checksum_of (selected_heal (\ _ => 0) seed) |> _assert_eq (checksum_of (zero_weights ()))

.c (c)
__assert_eq { expected = 0 }
.rs (rust)
__assert_eq { expected = 0 }

0